## Custom Retriever
https://reference.langchain.com/python/langchain-core/retrievers/BaseRetriever

Podemos definir el retriever de la forma que queramos; de esta forma, podemos obtener un retrieval más rico y dotar de más y mejor contexto a la LLM.

En este caso, el Retriever que definimos devuelve, de cada chunk con mayores scores, el chunk del header padre del que viene y sus vecinos

In [1]:
import psycopg2
from typing import List, Tuple, Dict
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.embeddings import Embeddings

class CustomRetriever(BaseRetriever):
    """
    Este Retriever personalizado devuelve, de cada chunk recuperado por la distancia del coseno, 
    los vecinos a este chunk y el header padre del que viene

    Args:
        coleccion_objetivo: Nombre de la colección que contiene los chunks
        embeddings: El modelo de embedding que usaremos para vectorizar la query
        connection: Las credenciales para acceder a la base de datos PostgreSQL
        k: El número de documentos que recuperaremos por la métrica cosine_distance
        neighbours: Cuantos vecinos (a los lados) queremos del resultado principal
    """
    coleccion_objetivo: str
    embeddings: Embeddings
    connection: Dict
    k: int = 3
    neighbours: int = 1 

    def _run_query(
        self,
        query: str,
        include_score: bool
    ) -> List[Document]:
        
        # Convertimos la query a embedding mediante el modelo de embedding
        query_vector = self.embeddings.embed_query(query) # Vectorizamos la query
        query_vector = str(query_vector) # PGVector espera que lo que le pasemos sea string

        # Definimos la consulta SQL
        sql_query = f"""
        
        -- CREAMOS UNA TABLA AUXILIAR QUE ME FILTRE LA 'TABLA' QUE QUEREMOS
        WITH coleccion_objetivo AS (
            select 
                uuid
            from langchain_pg_collection
            where name = %s
        ),
        -- CREAMOS UNA TABLA AUXILIAR A PARTIR DE LA METADATA. ESTA TABLA CONTENDRÁ LOS CHUNKS MÁS SIMILARES
        top_chunks AS (
            SELECT
                d.file_name,
                d.chunk_id,
                d.headers
            FROM langchain_pg_embedding AS d
            INNER JOIN coleccion_objetivo AS co
                ON d.collection_id = co.uuid
            ORDER BY embedding <=> %s::vector
            LIMIT %s
        )
        SELECT DISTINCT 
            d.id AS id,
            d.document AS page_content,
            d.cmetadata AS metadata
            {", d.embedding <=> %s::vector AS score" if include_score else ""}
        FROM langchain_pg_embedding d
        INNER JOIN coleccion_objetivo AS co
            ON d.collection_id = co.uuid
        INNER JOIN top_chunks tc 
            ON d.file_name = tc.file_name
        WHERE 
                                    -- CONDICIÓN DE PADRE --
            d.headers = (tc.headers - (-1)) and jsonb_array_length(tc.headers) > 1 --  Si jsonb_array_length(tc.headers) = 1, entonces no tiene padre; no perdemos el tiempo buscando quién es el padre
            OR
                                    -- CONDICIÓN DE VECINO --
            d.chunk_id between tc.chunk_id - %s AND tc.chunk_id + %s
        ORDER BY 
            {"score" if include_score else "d.id"};
        """

        params = (
            self.coleccion_objetivo,
            query_vector,
            self.k,
            *([query_vector] if include_score else []),
            self.neighbours,
            self.neighbours
        )
        try:
            # Nos conectamos a la base de datos
            with psycopg2.connect(**self.connection) as conn:
                # Nos conectamos para hacer la consulta a la base de datos
                with conn.cursor() as cur:
                    cur.execute(sql_query, params)
                    # Recuperamos todos los resultados. El resultado es una lista de tuplas donde
                    # cada elemento de la lista es una fila, y cada elemento de la tupla es una columna
                    return cur.fetchall()
        except Exception as e:
            print(f"Error al ejectura la consulta: {e}")
            return []



    # Langchain solo nos exige que cualquier retriever tenga el método _get_relevant_documents y que devuelva una lista de Documents
    def _get_relevant_documents(
        self, 
        query: str, 
        *, 
        run_manager=None
    ) -> List[Document]:

        docs = []
        results = self._run_query(query, include_score=False)
        for fila in results:
            id = fila[0]
            page_content = fila[1]
            metadata = fila[2]
            doc = Document(page_content, id=id, metadata=metadata)
            docs.append(doc)
        return docs
    
    # Esta es una función auxiliar que me permite saber qué tan bien me está recuperando los documentos
    def _get_relevant_documents_with_score(
        self, 
        query: str, 
        *, 
        run_manager=None
    ) -> List[Tuple[Document, float]]:
        
        docs = []
        results = self._run_query(query, include_score = True)
        for fila in results:
            id = fila[0]
            page_content = fila[1]
            metadata = fila[2]
            score = fila[3]
            doc = Document(page_content, id=id, metadata=metadata)
            docs.append(tuple([doc, score]))
        return docs

# INSTANCIAMOS EL CUSTOM RETRIEVER

In [3]:
from config import config # Importamos esta función que simplemente me lee las credenciales de acceso a la base de datos
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY1")

query_embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001", 
    google_api_key = api_key,
    task_type="RETRIEVAL_QUERY",
    output_dimensionality=1024
) 

retriever = CustomRetriever(
    coleccion_objetivo = "urbo_doc",
    embeddings = query_embedding,
    connection = config(),
    k = 20,
    neighbours = 5
)

# Recuperación con Chain

Langchain nos permite usar un operador, | , que encadena funciones (en Langchain, Runnables), es decir, (f | g)(x) = f(g(x)) . De esta forma no tenemos que hacer todo manualmente, sino que el output de una Runnable es el input de la siguiente

In [4]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

def format_docs(docs):
    return ("\n\n" + "-"*50+"\n\n").join([doc.page_content for doc in docs])

SYSTEM_TEMPLATE = """
Eres un asistente experto en extracción de información. Tu objetivo es responder preguntas basándote EXCLUSIVAMENTE en el CONTEXTO proporcionado.

REGLAS CRÍTICAS DE COMPORTAMIENTO:
1. **Fidelidad Estricta**: Responde SOLO con la información presente en el contexto. 
2. **Protocolo de Ausencia**: Si la respuesta no se encuentra en el contexto, o si el contexto no contiene suficiente información para responder de forma completa, responde exactamente: "Lo siento, pero no tengo información suficiente en los documentos para responder a esa pregunta."
3. **Prohibición de Conocimiento Externo**: No utilices tus propios conocimientos previos ni asumas hechos que no estén escritos en el texto proporcionado.
"""

USER_TEMPLATE = """
**CONTEXTO**:
{context}

---

**PREGUNTA**:
{query}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("user", USER_TEMPLATE)
])

llm = ChatGoogleGenerativeAI(
    google_api_key = api_key,
    model="gemini-2.5-flash",
    temperature = 0
)

str_parser = StrOutputParser() # Esto nos devuelve la respuesta de la LLM en un str (nos evita estar haciendo constantemente .content)


chain = ({"query": RunnablePassthrough(), # Esto es lo mismo que la función identidad
         "context": retriever | RunnableLambda(format_docs)} # Envolvemos la función de python en un Runnable
        | prompt | llm | str_parser)



In [5]:
query = """
¿Cómo se instala la aplicación Urbo?
"""

response = chain.invoke(query)
print(response)

La aplicación Urbo se puede instalar de varias maneras, según el entorno deseado:

**1. Instalación con Docker:**
*   Se puede construir un entorno de trabajo operativo con URBO DEPLOYER y URBO BUILDER utilizando el `Dockerfile` incluido en el repositorio, ejecutando:
    ```
    docker build --rm -t urbodeployer:latest .
    ```
*   Alternativamente, el repositorio incluye un fichero `docker-compose.yaml`. Al lanzar el stack con `docker-compose up -d`, se levantarán:
    *   Un servicio web para ejecutar builder *as a service* accesible a través del puerto 6000 (`http://localhost:6000`).
    *   Un servicio web para ejecutar encryptor *as a service* accesible a través del puerto 5000 (`http://localhost:5000`).

**2. Instalación con máquina virtual VirtualBox:**
*   Se proporciona un fichero OVA de la máquina virtual llamado `debian_urbodeployer.ova`.
*   **Requisitos:** Tener instalado `VM VirtualBox Extension Pack`.
*   **Pasos para importar:**
    1.  Archivo -> Importar servicio vi

In [7]:
retriever._get_relevant_documents_with_score(query)[:10]

[(Document(id='390', metadata={'headers': ['Instalación con máquina virtual VirtualBox'], 'chunk_id': 1, 'file_name': 'installation-vm.md'}, page_content='# Instalación con máquina virtual VirtualBox  \nFichero OVA de la máquina virtual (en SharePoint) [debian_urbodeployer.ova](https://telefonicacorp.sharepoint.com/:u:/s/smartcitiesredes/EdtZIOV2UsBMqJx2vErGqDIBeGIjCccnk9k2Oeyjb3HgCg?e=FbOeTv) (MDDSUM: `d9c1f8f0d63cd55ca9253f9e2afd2cc8`)  \nInstrucciones para importar en VirtualBox:  \n* Requisito: Tener instalado `VM VirtualBox Extension Pack`\n* Archivo -> Importar servicio virtualizado\n* Marcar: "Generar nuevas direcciones MAC para todos los adaptadores de red"\n* Cambiar la tarjeta de red por una del propio sistema host o configurar el modo `NAT`  \nCredenciales:  \n* Usuario: techtransfer\n* pass: 4dmin_urbo  \nSoftware instalado:  \n* Python\n* GO\n* CUE\n* VS Code  \nRepositorios clonados:  \n* Ruta del código urbo-deployer: `/home/Documentos/TEF/code/urbo-deployer`\n* El repo 